# TIỀN XỬ LÝ & LÀM SẠCH DỮ LIỆU CẢNH BÁO HỌC TẬP (`Tienxl.ipynb`)

Quy trình thực hiện:
1. **Đọc 2 file CSV gốc** từ thư mục `datasets/raw`:
   - `DS_HOCTIEP_da_gan_nhan_hoc_luc_canh_bao.csv`
   - `DS_HOCTIEP_GOP_2_FILE_DA_CHUAN_HOA.csv`
2. **Làm sạch & Chuẩn hóa dữ liệu**:
   - Bỏ 2 dòng tiêu đề thừa ở đầu file (`header=2`), chuẩn hóa tên cột.
   - Đồng nhất cấu trúc cột giữa 2 file (`Ngành`, `Khóa`, `Kỳ xét`, `TBC HK`, `TBC TL`, `Số TC Đạt`, `Số TC F`, `Học lực`, `Cảnh báo`), loại bỏ cột `Lớp` bị khuyết >84% dữ liệu.
   - Chuẩn hóa chuỗi ký tự và đồng nhất tên ngành bị lệch chữ hoa/thường (`Công nghệ kỹ thuật Cơ khí` -> `Công nghệ kỹ thuật cơ khí`).
   - Ép kiểu dữ liệu số, loại bỏ các dòng `NaN`, kiểm tra miền giá trị hợp lệ (`0 <= TBC HK, TBC TL <= 4`) và loại bỏ các bản ghi lỗi (toàn bộ chỉ số học tập bằng 0).
   - Gộp 2 file và khử hoàn toàn các bản ghi trùng lặp.
3. **Chia sẵn tập dữ liệu 80% Train – 20% Test (Stratified Split)**:
   - Gắn trực tiếp cột `split` (`'train'` / `'test'`) vào dữ liệu đã làm sạch theo tỷ lệ 80% Train và 20% Test (phân tầng theo `Học lực` và `Cảnh báo`).
4. **Lưu trữ vào `datasets/processed`**:
   - Xuất ra 2 định dạng: **`cleaned_data.parquet`** và **`cleaned_data.json`**.

In [1]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    candidate_roots = ['/content/drive/MyDrive/FMNN_CODE/FMN_EduWarning']
except ImportError:
    candidate_roots = [
        os.path.abspath(os.path.join(os.getcwd(), '..')),
        os.path.abspath(os.path.join(os.getcwd(), '..', 'FMN_EduWarning')),
        os.path.abspath(os.path.join(os.getcwd(), 'FMN_EduWarning')),
        os.path.abspath(os.getcwd())
    ]

project_root = next((p for p in candidate_roots if os.path.exists(os.path.join(p, 'datasets', 'raw'))), candidate_roots[0])
raw_data_path = os.path.join(project_root, 'datasets', 'raw')
processed_data_path = os.path.join(project_root, 'datasets', 'processed')
os.makedirs(processed_data_path, exist_ok=True)

In [2]:
# 2. Đọc 2 file CSV từ thư mục datasets/raw (bỏ 2 dòng tiêu đề thừa ở đầu với header=2)
csv_file_1 = os.path.join(raw_data_path, 'DS_HOCTIEP_da_gan_nhan_hoc_luc_canh_bao.csv')
csv_file_2 = os.path.join(raw_data_path, 'DS_HOCTIEP_GOP_2_FILE_DA_CHUAN_HOA.csv')

df_raw_1 = pd.read_csv(csv_file_1, encoding='utf-8-sig', header=2)
df_raw_2 = pd.read_csv(csv_file_2, encoding='utf-8-sig', header=2)

# Xóa khoảng trắng dư thừa ở tên cột
df_raw_1.columns = df_raw_1.columns.str.strip()
df_raw_2.columns = df_raw_2.columns.str.strip()

In [3]:
# 3. Làm sạch và chuẩn hóa dữ liệu từ 2 file CSV
COMMON_COLS = ['Ngành', 'Khóa', 'Kỳ xét', 'TBC HK', 'TBC TL', 'Số TC Đạt', 'Số TC F', 'Học lực', 'Cảnh báo']
FLOAT_COLS = ['TBC HK', 'TBC TL']
INT_COLS = ['Kỳ xét', 'Số TC Đạt', 'Số TC F', 'Học lực', 'Cảnh báo']

# Gộp dữ liệu từ 2 file theo đúng thứ tự cột chuẩn
df_combined = pd.concat([df_raw_2[COMMON_COLS], df_raw_1[COMMON_COLS]], ignore_index=True)
initial_count = len(df_combined)

# 3.1. Chuẩn hóa chuỗi ký tự và đồng nhất tên ngành bị viết lệch hoa/thường
df_combined['Ngành'] = (
    df_combined['Ngành']
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .replace({'Công nghệ kỹ thuật Cơ khí': 'Công nghệ kỹ thuật cơ khí'})
)
df_combined['Khóa'] = df_combined['Khóa'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

# 3.2. Ép kiểu dữ liệu số và loại bỏ các dòng chứa NaN
for col in FLOAT_COLS + INT_COLS:
    df_combined[col] = pd.to_numeric(df_combined[col], errors='coerce')

after_nan_df = df_combined.dropna(subset=COMMON_COLS).copy()
nan_removed = initial_count - len(after_nan_df)

for col in INT_COLS:
    after_nan_df[col] = after_nan_df[col].astype(int)

# 3.3. Kiểm tra miền giá trị hợp lệ và loại bỏ các dòng lỗi (toàn bộ chỉ số học tập bằng 0)
valid_mask = (
    (after_nan_df['TBC HK'] >= 0.0) & (after_nan_df['TBC HK'] <= 4.0) &
    (after_nan_df['TBC TL'] >= 0.0) & (after_nan_df['TBC TL'] <= 4.0) &
    (after_nan_df['Số TC Đạt'] >= 0) &
    (after_nan_df['Số TC F'] >= 0) &
    (after_nan_df['Kỳ xét'] >= 1) &
    (after_nan_df['Học lực'].isin([1, 2, 3, 4, 5, 6])) &
    (after_nan_df['Cảnh báo'].isin([0, 1])) &
    ~(
        (after_nan_df['TBC HK'] == 0.0) &
        (after_nan_df['TBC TL'] == 0.0) &
        (after_nan_df['Số TC Đạt'] == 0) &
        (after_nan_df['Số TC F'] == 0)
    )
)
valid_df = after_nan_df[valid_mask].copy()
invalid_removed = len(after_nan_df) - len(valid_df)

# 3.4. Khử trùng lặp dữ liệu giữa 2 file và đánh lại STT
df_cleaned = valid_df.drop_duplicates(subset=COMMON_COLS).reset_index(drop=True)
duplicates_removed = len(valid_df) - len(df_cleaned)
df_cleaned.insert(0, 'STT', np.arange(1, len(df_cleaned) + 1))

In [4]:
# 4. Chia sẵn dữ liệu theo tỷ lệ 80% Train và 20% Test (Stratified theo Học lực & Cảnh báo)
stratify_key = df_cleaned['Học lực'].astype(str) + '_' + df_cleaned['Cảnh báo'].astype(str)

train_idx, test_idx = train_test_split(
    df_cleaned.index,
    test_size=0.20,
    random_state=42,
    stratify=stratify_key
)

df_cleaned['split'] = 'train'
df_cleaned.loc[test_idx, 'split'] = 'test'

n_train = int((df_cleaned['split'] == 'train').sum())
n_test = int((df_cleaned['split'] == 'test').sum())

# 5. Lưu dữ liệu sau khi làm sạch & chia tập ra 2 file Parquet và JSON trong datasets/processed
parquet_output_path = os.path.join(processed_data_path, 'cleaned_data.parquet')
json_output_path = os.path.join(processed_data_path, 'cleaned_data.json')

df_cleaned.to_parquet(parquet_output_path, index=False)
df_cleaned.to_json(json_output_path, orient='records', force_ascii=False, indent=2)

# 6. Hiển thị bảng tổng hợp các mốc xử lý theo mẫu
print(f"Đã lưu thành công! Kích thước bảng sạch: {df_cleaned.shape}")

df_report = pd.DataFrame([
    {
        'moc_xu_ly': '1. Dữ liệu gốc (raw)',
        'mo_ta': f'Gộp 2 file CSV gốc từ datasets/raw ({len(df_raw_1)} + {len(df_raw_2)} dòng)',
        'so_dong': initial_count
    },
    {
        'moc_xu_ly': '2. Sau lọc khuyết (NaN) & chuẩn hóa ngành',
        'mo_ta': f'Bỏ cột Lớp khuyết >84%, loại {nan_removed} dòng NaN, đồng nhất {df_cleaned["Ngành"].nunique()} ngành',
        'so_dong': len(after_nan_df)
    },
    {
        'moc_xu_ly': '3. Sau lọc ngoại lai / sai miền giá trị',
        'mo_ta': f'Loại {invalid_removed} dòng lỗi toàn bộ chỉ số học tập bằng 0',
        'so_dong': len(valid_df)
    },
    {
        'moc_xu_ly': '4. Sau loại trùng bản ghi',
        'mo_ta': f'Loại {duplicates_removed} dòng trùng lặp giữa 2 file CSV',
        'so_dong': len(df_cleaned)
    },
    {
        'moc_xu_ly': '5. Tập sạch cuối cùng',
        'mo_ta': 'Giữ đầy đủ các thuộc tính học tập chuẩn và nhãn phân loại',
        'so_dong': len(df_cleaned)
    },
    {
        'moc_xu_ly': 'Tập Train (80%)',
        'mo_ta': f'Huấn luyện ({n_train / len(df_cleaned) * 100:.1f}%)',
        'so_dong': n_train
    },
    {
        'moc_xu_ly': 'Tập Test (20%)',
        'mo_ta': f'Kiểm tra ({n_test / len(df_cleaned) * 100:.1f}%)',
        'so_dong': n_test
    }
])
df_report

Đã lưu thành công! Kích thước bảng sạch: (24963, 11)


,moc_xu_ly,mo_ta,so_dong
0,1. Dữ liệu gốc (raw),Gộp 2 file CSV gốc từ datasets/raw (21878 + 25985 dòng),47863
1,2. Sau lọc khuyết (NaN) & chuẩn hóa ngành,"Bỏ cột Lớp khuyết >84%, loại 0 dòng NaN, đồng nhất 47 ngành",47863
2,3. Sau lọc ngoại lai / sai miền giá trị,Loại 4 dòng lỗi toàn bộ chỉ số học tập bằng 0,47859
3,4. Sau loại trùng bản ghi,Loại 22896 dòng trùng lặp giữa 2 file CSV,24963
4,5. Tập sạch cuối cùng,Giữ đầy đủ các thuộc tính học tập chuẩn và nhãn phân loại,24963
5,Tập Train (80%),Huấn luyện (80.0%),19970
6,Tập Test (20%),Kiểm tra (20.0%),4993
